# 03 · Join Sofascore + Capology — Turkey Süper Lig 20/21

Integración de estadísticas de rendimiento (Sofascore) con datos salariales (Capology)
para la temporada **2020/21 de Süper Lig turca**.

**Flujo de matching:**
1. Normalización de nombres (tildes, mayúsculas, caracteres especiales)
2. `TEAM_MAP`: alineación manual de nombres de equipo entre fuentes
3. Merge exacto normalizado
4. Fuzzy matching en cuatro niveles:
   - Score ≥ 0.90 → aceptación automática
   - 0.75 ≤ score < 0.90 → revisión manual
   - 0.50 ≤ score < 0.75 → revisión manual estricta
   - score < 0.50 → revisión manual muy estricta
5. Revisión de jugadores sin salario
6. Guardado en `data/master/`

---

## 1. Imports y rutas

In [36]:
import pandas as pd
import unicodedata
import re
from rapidfuzz import fuzz
from pathlib import Path
from IPython.display import display

ROOT       = Path.cwd().parents[1]
SF_DIR     = ROOT / 'data' / 'processed' / 'sofascore'
CG_DIR     = ROOT / 'data' / 'processed' / 'capology'
MASTER_DIR = ROOT / 'data' / 'master'
MASTER_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Rutas configuradas')
print(f'   Root:   {ROOT}')
print(f'   Master: {MASTER_DIR}')

✅ Rutas configuradas
   Root:   d:\USER\Desktop\TFM
   Master: d:\USER\Desktop\TFM\data\master


## 2. Función de normalización

In [37]:
def normalize(s):
    """
    Normaliza un string para comparación: elimina tildes, pasa a minúsculas,
    elimina caracteres especiales y espacios extra.
    """
    if pd.isna(s):
        return ''
    s = str(s)
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('ascii')
    s = re.sub(r'[^a-z0-9\s]', ' ', s.lower().strip())
    return re.sub(r'\s+', ' ', s).strip()

print('✅ Función definida')

✅ Función definida


## 3. Carga de datos

In [38]:
df_sf = pd.read_csv(SF_DIR / 'df_turkey_2021.csv').copy()
df_cg = pd.read_csv(CG_DIR / 'cg_turkey_2021.csv').copy()

print(f'Sofascore:  {df_sf.shape[0]} jugadores | {df_sf.shape[1]} columnas')
print(f'Capology:   {df_cg.shape[0]} jugadores | {df_cg.shape[1]} columnas')

Sofascore:  660 jugadores | 116 columnas
Capology:   737 jugadores | 9 columnas


## 4. Normalización

In [39]:
df_sf['player_norm'] = df_sf['player'].apply(normalize)
df_sf['team_norm']   = df_sf['team'].apply(normalize)
df_cg['player_norm'] = df_cg['player'].apply(normalize)
df_cg['team_norm']   = df_cg['club'].apply(normalize)

print('✅ Normalización aplicada')

✅ Normalización aplicada


## 5. Alineación de equipos (TEAM_MAP)

### 5.1 Identificar discrepancias de nombres de equipo

In [40]:
solo_sf = set(df_sf['team_norm'].unique()) - set(df_cg['team_norm'].unique())
solo_cg = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())

print('En Sofascore pero no en Capology:')
for e in sorted(solo_sf): print(f'   {e}')
print()
print('En Capology pero no en Sofascore:')
for e in sorted(solo_cg): print(f'   {e}')

En Sofascore pero no en Capology:
   basaksehir fk
   besiktas jk
   erzurumspor fk
   fatih karagumruk
   gaziantep fk
   genclerbirligi
   kasmpasa
   mke ankaragucu

En Capology pero no en Sofascore:
   ankaragucu
   basaksehir
   besiktas
   erzurumspor
   gaziantep bb
   genclerbirligi sk
   karagumrukspor
   kasimpasa


### 5.2 Aplicar TEAM_MAP

Rellenar con las discrepancias identificadas en la celda anterior.

In [41]:
# ── Ajustar según la celda anterior ──────────────────────────
TEAM_MAP = {'ankaragucu':'mke ankaragucu',
            'basaksehir':'basaksehir fk',
            'besiktas':'besiktas jk',
            'erzurumspor':'erzurumspor fk',
            'gaziantep bb':'gaziantep fk',
            'genclerbirligi sk':'genclerbirligi',
            'karagumrukspor':'fatih karagumruk',
            'kasimpasa':'kasmpasa'
}
# ─────────────────────────────────────────────────────────────

df_cg['team_norm'] = df_cg['team_norm'].replace(TEAM_MAP)

diff = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())
if diff:
    print(f'⚠️  Equipos de CG aún sin match en SF: {diff}')
else:
    print('✅ Todos los equipos alineados')


✅ Todos los equipos alineados


## 6. Merge exacto normalizado

In [42]:
df_merged = df_sf.merge(
    df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
            'position', 'age', 'nationality']],
    on=['player_norm', 'team_norm'],
    how='left'
)

matched = df_merged['gross_annual_eur'].notna().sum()
total   = len(df_merged)

print(f'Merge exacto: {matched}/{total} ({matched/total:.1%})')
print(f'Sin emparejar: {total - matched}')

Merge exacto: 538/660 (81.5%)
Sin emparejar: 122


## 7. Fuzzy matching sobre los no emparejados

Se generan candidatos para todos los jugadores sin match exacto,
sin umbral mínimo, y se clasifican en cuatro niveles.

In [43]:
df_unmatched = df_merged[df_merged['gross_annual_eur'].isna()].copy()
cg_by_team   = df_cg.groupby('team_norm')['player_norm'].apply(list).to_dict()

rows = []
for _, row in df_unmatched[['player','team','player_norm','team_norm']].drop_duplicates().iterrows():
    candidates = cg_by_team.get(row['team_norm'], [])
    best_match, best_score = None, 0
    for cand in candidates:
        score = fuzz.ratio(row['player_norm'], cand) / 100
        if score > best_score:
            best_score = score
            best_match = cand
    if best_match is not None:
        rows.append({
            'player_sf'  : row['player'],
            'team'       : row['team'],
            'player_norm': row['player_norm'],
            'team_norm'  : row['team_norm'],
            'cg_match'   : best_match,
            'score'      : round(best_score, 3)
        })

df_candidates = pd.DataFrame(rows).sort_values('score', ascending=False)
auto_matches     = df_candidates[df_candidates['score'] >= 0.90].copy()
review_matches   = df_candidates[(df_candidates['score'] >= 0.75) & (df_candidates['score'] < 0.90)].copy()
low_matches      = df_candidates[(df_candidates['score'] >= 0.50) & (df_candidates['score'] < 0.75)].copy()
very_low_matches = df_candidates[df_candidates['score'] < 0.50].copy()

print(f'Auto-aceptados    (score ≥ 0.90):          {len(auto_matches)}')
print(f'Revisión media    (0.75 ≤ score < 0.90):   {len(review_matches)}')
print(f'Revisión estricta (0.50 ≤ score < 0.75):   {len(low_matches)}')
print(f'Revisión muy est. (score < 0.50):           {len(very_low_matches)}')

Auto-aceptados    (score ≥ 0.90):          59
Revisión media    (0.75 ≤ score < 0.90):   18
Revisión estricta (0.50 ≤ score < 0.75):   30
Revisión muy est. (score < 0.50):           15


### 7.1 Matches automáticos (score ≥ 0.90)

Revisar para confirmar que todos son correctos.

In [44]:
auto_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
59,Dimitris Chatziisaias,Çaykur Rizespor,dimitrios chatziisaias,0.977
38,Dimitris Kolovetsios,Kayserispor,dimitrios kolovetsios,0.976
56,Alexander Søderlund,Çaykur Rizespor,alexander soderlund,0.973
15,Abdülkerim Bardakcı,Konyaspor,abdulkerim bardakci,0.973
63,Kubilay Kanatsızkuş,Yeni Malatyaspor,kubilay kanatsizkus,0.973
53,Radosław Murawski,Denizlispor,radoslaw murawski,0.970
87,Mevlut Han Ekelik,Antalyaspor,mevluthan ekelik,0.970
84,Bilal Başaçıkoğlu,Gaziantep FK,bilal basacikoglu,0.970
30,Ertuğrul Taşkıran,Kasımpaşa,ertugrul taskiran,0.970
7,Taylan Antalyalı,Galatasaray,taylan antalyali,0.968


### 7.2 Revisión media (0.75 ≤ score < 0.90)

Añadir a `EXCLUDE_FROM_FUZZY` el `player_norm` de los incorrectos.

In [45]:
review_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
1,Muammer Yıldırım,Sivasspor,muammer yildirim,0.897
98,Volkan Fındıklı,Konyaspor,volkan findikli,0.889
55,Barış Yardımcı,Konyaspor,baris yardimci,0.880
108,Sami Gokhan Altiparmak,Gençlerbirliği,gokhan altiparmak,0.872
10,Serkan Kırıntılı,Alanyaspor,serkan kirintili,0.857
39,Mbwana Ally Samatta,Fenerbahçe,mbwana samatta,0.848
4,Aaron-Salem Boupendza,Hatayspor,aaron boupendza,0.833
116,Ahmet Oytun Özdoğan,Fenerbahçe,oytun ozdogan,0.812
114,Çağlar Şahin Akbaba,Gaziantep FK,caglar akbaba,0.812
12,Berat Ayberk Özdemir,Trabzonspor,berat ozdemir,0.788


In [46]:
# ── Falsos positivos a excluir del nivel medio ────────────────
EXCLUDE_FROM_FUZZY = [

]
# ─────────────────────────────────────────────────────────────

review_accepted = review_matches[~review_matches['player_norm'].isin(EXCLUDE_FROM_FUZZY)]
print(f'Aceptados: {len(review_accepted)} | Excluidos: {len(EXCLUDE_FROM_FUZZY)}')


Aceptados: 18 | Excluidos: 0


### 7.3 Revisión estricta (0.50 ≤ score < 0.75)

Por defecto ninguno se acepta. Añadir a `ACCEPT_LOW_FUZZY` los correctos.

In [47]:
low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
95,Muhammed Eren Kiryolcu,Denizlispor,eren kiryolcu,0.743
18,Eren Elmalı,Kasımpaşa,evren eren elmali,0.741
70,Mehmet Murat Uçar,Erzurumspor FK,murat ucar,0.741
103,Hüseyin Atakan Üner,Beşiktaş JK,atakan uner,0.733
94,Emre Karaal,Başakşehir FK,emre kaplan,0.727
52,Jefferson Junior,Gaziantep FK,jefferson,0.720
14,Campanharo,Kayserispor,gustavo campanharo,0.714
27,Guilherme Marques,Göztepe,guilherme,0.692
82,Mehmet Feyzi Yıldırım,Kasımpaşa,feyzi yildirim,0.688
111,Huseyin Furkan Uslu,Denizlispor,huseyin altintas,0.629


In [48]:
# ── Matches de score bajo confirmados manualmente ─────────────
ACCEPT_LOW_FUZZY = ['muhammed eren kiryolcu',
                    'eren elmal',
                    'mehmet murat ucar',
                    'huseyin atakan uner',
                    'jefferson junior',
                    'campanharo',
                    'guilherme marques',
                    'mehmet feyzi yldrm',
                    'amilton da silva',
                    'andre balada',
                    'radamel falcao',
                    'carlos ponck',
                    'guilherme haubert sitya',
                    'alan carius',
                    'munir el kajoui'
                    
                    

]
# ─────────────────────────────────────────────────────────────

low_accepted = low_matches[low_matches['player_norm'].isin(ACCEPT_LOW_FUZZY)]
print(f'Aceptados del nivel bajo: {len(low_accepted)}')


Aceptados del nivel bajo: 15


### 7.4 Revisión muy estricta (score < 0.50)

Por defecto ninguno se acepta. Añadir a `ACCEPT_VERY_LOW_FUZZY` los correctos.

In [49]:
very_low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
8,Kağan Moradaoğlu,Trabzonspor,bilal basacikoglu,0.485
42,Hasan Yesilyurt,Kasımpaşa,tarkan serbest,0.483
115,Emre Yildrim,Denizlispor,eren kiryolcu,0.480
112,Fatih Yiğit Şanlıtürk,Fenerbahçe,bright osayi samuel,0.462
117,Bartuğ Elmaz,Galatasaray,arda turan,0.455
0,Mario Šitum,Kayserispor,karlo muhar,0.455
28,Michael Frey,Fenerbahçe,omer beyaz,0.455
58,İsmail Yüksek,Fenerbahçe,papiss demba cisse,0.452
96,Mert Sarikus,Denizlispor,muris mesanovic,0.444
97,Umut Nayir,Beşiktaş JK,muhayer oktay,0.435


In [50]:
# ── Matches very low confirmados manualmente ──────────────────
ACCEPT_VERY_LOW_FUZZY = [

]
# ─────────────────────────────────────────────────────────────

very_low_accepted = very_low_matches[very_low_matches['player_norm'].isin(ACCEPT_VERY_LOW_FUZZY)]
print(f'Aceptados del nivel very low: {len(very_low_accepted)}')


Aceptados del nivel very low: 0


### 7.5 Aplicar todos los fuzzy matches aceptados

In [51]:
all_fuzzy    = pd.concat([auto_matches, review_accepted, low_accepted, very_low_accepted], ignore_index=True)
fuzzy_lookup = dict(zip(all_fuzzy['player_norm'], all_fuzzy['cg_match']))

df_merged['player_norm_fuzzy'] = df_merged.apply(
    lambda r: fuzzy_lookup.get(r['player_norm'], r['player_norm'])
    if pd.isna(r['gross_annual_eur']) else r['player_norm'],
    axis=1
)

df_final = (
    df_merged
    .drop(columns=['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality'])
    .merge(
        df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
               'position', 'age', 'nationality']],
        left_on=['player_norm_fuzzy', 'team_norm'],
        right_on=['player_norm', 'team_norm'],
        how='left'
    )
    .drop(columns=['player_norm_y', 'player_norm_fuzzy'])
    .rename(columns={'player_norm_x': 'player_norm'})
)

matched_final = df_final['gross_annual_eur'].notna().sum()
print(f'Resultado final: {matched_final}/{len(df_final)} ({matched_final/len(df_final):.1%})')
print(f'Sin salario:     {len(df_final) - matched_final}')

Resultado final: 630/660 (95.5%)
Sin salario:     30


## 8. Revisión de jugadores sin salario

Ordenados por equipo y minutos jugados para identificar si alguno debería tener salario.

In [52]:
sin_salario = (
    df_final[df_final['gross_annual_eur'].isna()]
    [['player', 'team', 'minutesPlayed', 'appearances', 'goals', 'assists']]
    .sort_values(['team', 'minutesPlayed'], ascending=[True, False])
    .reset_index(drop=True)
)

pd.set_option('display.max_rows', None)
print(f'Total sin salario: {len(sin_salario)}')
display(sin_salario)
pd.reset_option('display.max_rows')

Total sin salario: 30


,player,team,minutesPlayed,appearances,goals,assists
0,Yusuf Özdemir,Alanyaspor,10,1,0,0
1,Muhammed Arda Uzun,Alanyaspor,1,1,0,0
2,Paul Mukairu,Antalyaspor,135,3,0,0
3,Emre Karaal,Başakşehir FK,8,1,0,0
4,Souza,Beşiktaş JK,2880,33,2,3
5,Umut Nayir,Beşiktaş JK,28,2,0,0
6,Emirhan Kascioglu,Denizlispor,147,3,0,0
7,Emre Yildrim,Denizlispor,90,1,0,0
8,Mehmet Ali Ulaman,Denizlispor,45,1,0,0
9,Mert Sarikus,Denizlispor,8,1,0,0


### 8.1 Comparación manual por equipo

Para cada equipo con jugadores sin salario se muestra la plantilla completa de Capology
ordenada alfabéticamente por nombre normalizado, facilitando la detección visual de matches fallidos.

In [53]:
equipos_sin_salario = sin_salario['team'].unique()

for equipo in sorted(equipos_sin_salario):
    sf_jugadores = sin_salario[sin_salario['team'] == equipo][['player', 'minutesPlayed']].sort_values('player')

    equipo_norm  = normalize(equipo)
    cg_jugadores = (
        df_cg[df_cg['team_norm'] == equipo_norm][['player', 'player_norm']]
        .sort_values('player_norm')
        .reset_index(drop=True)
    )

    print(f'\n{"="*60}')
    print(f'  {equipo}  —  SF sin salario:')
    display(sf_jugadores.reset_index(drop=True))
    print(f'  CG plantilla completa:')
    display(cg_jugadores)


  Alanyaspor  —  SF sin salario:


,player,minutesPlayed
0,Muhammed Arda Uzun,1
1,Yusuf Özdemir,10


  CG plantilla completa:


,player,player_norm
0,Adam Bareiro,adam bareiro
1,Ahmet Cagri Güney,ahmet cagri guney
2,Ahmet Gülay,ahmet gulay
3,Alpay Celebi,alpay celebi
4,Anastasios Bakasetas,anastasios bakasetas
5,Berkan Kutlu,berkan kutlu
6,Ceyhun Gülselam,ceyhun gulselam
7,Damian Kadzior,damian kadzior
8,Davidson,davidson
9,Efecan Karaca,efecan karaca



  Antalyaspor  —  SF sin salario:


...

,player,player_norm
0,Abdülkadir Ömür,abdulkadir omur
1,Abdulkadir Parmak,abdulkadir parmak
2,Ahmet Canbaz,ahmet canbaz
3,Ahmetcan Kaplan,ahmetcan kaplan
4,Anastasios Bakasetas,anastasios bakasetas
5,Anders Trondsen,anders trondsen
6,Anthony Nwakaeme,anthony nwakaeme
7,Arda Akbulut,arda akbulut
8,Atakan Gündüz,atakan gunduz
9,Benik Afobe,benik afobe



  Çaykur Rizespor  —  SF sin salario:


,player,minutesPlayed
0,Nadir Çiftçi,25


  CG plantilla completa:


,player,player_norm
0,Abdullah Durak,abdullah durak
1,Alberk Koc,alberk koc
2,Alexander Söderlund,alexander soderlund
3,Aminu Umar,aminu umar
4,Bogachan Kazmaz,bogachan kazmaz
5,Braian Samudio,braian samudio
6,Damjan Djokovic,damjan djokovic
7,Dario Melnjak,dario melnjak
8,Dimitrios Chatziisaias,dimitrios chatziisaias
9,Dogan Erdogan,dogan erdogan


In [54]:
# ── Matches manuales (nombres muy distintos o traspasos invernales) ──
# Formato: (player_norm_sf, team_norm_sf): (player_norm_cg, team_norm_cg)
MANUAL_MATCHES = {
    ('souza', 'besiktas jk')                    : ('josef', 'besiktas jk'),
}
# ────────────────────────────────────────────────────────────────────
print(f'Matches manuales definidos: {len(MANUAL_MATCHES)}')


Matches manuales definidos: 1


In [55]:
# Aplicar matches manuales sobre los que siguen sin salario
for (p_sf, t_sf), (p_cg, t_cg) in MANUAL_MATCHES.items():
    mask = (df_final['player_norm'] == p_sf) & (df_final['team_norm'] == t_sf) & (df_final['gross_annual_eur'].isna())
    datos_cg = df_cg[(df_cg['player_norm'] == p_cg) & (df_cg['team_norm'] == t_cg)]
    if not datos_cg.empty and mask.any():
        for col in ['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality']:
            df_final.loc[mask, col] = datos_cg[col].values[0]
        print(f'✅ Match manual aplicado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')
    else:
        print(f'⚠️  No encontrado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')

matched_tras_manual = df_final['gross_annual_eur'].notna().sum()
print(f'\nTras matches manuales: {matched_tras_manual}/{len(df_final)} ({matched_tras_manual/len(df_final):.1%})')

✅ Match manual aplicado: souza (besiktas jk) → josef (besiktas jk)

Tras matches manuales: 631/660 (95.6%)


In [56]:
pd.reset_option('display.max_rows')

## 9. Guardado

Una vez revisado todo, se eliminan las columnas auxiliares y se guarda en `data/master/`.

In [57]:
df_final = df_final.drop(columns=['player_norm', 'team_norm'])

nombre_salida = 'master_turkey_2021.csv'
df_final.to_csv(MASTER_DIR / nombre_salida, index=False)

print(f'✅ Guardado: {nombre_salida}')
print(f'   Jugadores totales:  {len(df_final)}')
print(f'   Con salario:        {df_final["gross_annual_eur"].notna().sum()}')
print(f'   Sin salario (NaN):  {df_final["gross_annual_eur"].isna().sum()}')
print(f'   Columnas:           {df_final.shape[1]}')

✅ Guardado: master_turkey_2021.csv
   Jugadores totales:  660
   Con salario:        631
   Sin salario (NaN):  29
   Columnas:           121
